# ByteNet — FFT-75 Scenario #1 Benchmark

**Paper**: ByteNet: Rethinking Multimedia File Fragment Classification through Visual Perspectives (Liu et al., 2023)

**Frozen benchmark**: 512-byte fragments · 75 classes · official pre-split NPZ files

**Dataset source**: Google Drive (downloaded via gdown, extracted to `/kaggle/working/data/`)

Pipeline:
1. Install dependencies
2. Clone repo (`eval-ByteNet` branch)
3. Download & extract FFT-75 dataset from Google Drive  ← **single cell, run once**
4. Configure paths & verify dataset integrity
5. Sanity training (2 epochs, 2 000 samples)
6. Full training (50 epochs, warmup+cosine, AMP, CutMix/Mixup)
7. Evaluation on frozen test set
8. Zip and save outputs

> **Nothing to change before running** — all paths are pre-configured.


## Cell 1 — Install Dependencies

In [ ]:
import gc
import os
import subprocess
import sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Core scientific stack (Kaggle usually has these, but pin versions)
pip('pyyaml>=6.0')
pip('scikit-learn>=1.5')
pip('pandas>=2.2')
pip('numpy>=1.26')
pip('matplotlib>=3.9')
pip('seaborn>=0.13')
pip('tqdm>=4.66')
pip('gdown>=5.1')  # for Google Drive download

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM    : {props.total_memory / 1e9:.1f} GB')

import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f'RAM     : {ram_gb:.1f} GB')


## Cell 2 — Clone Repo

In [ ]:
import os
import shutil
from pathlib import Path

WORKING      = Path('/kaggle/working')
REPO_DIR     = WORKING / 'deepcarv'
BRANCH_NAME  = 'eval-ByteNet'

# For private repos: set GITHUB_TOKEN to a Personal Access Token with read access.
# For public repos: leave as empty string.
GITHUB_TOKEN = ''   # e.g. 'ghp_xxxxxxxxxxxx'

# Force fresh clone if src/ is missing (handles stale / wrong-branch clones)
if not REPO_DIR.exists() or not (REPO_DIR / 'src').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    if GITHUB_TOKEN:
        clone_url = f'https://{GITHUB_TOKEN}@github.com/yuvnahr/deepcarv.git'
    else:
        clone_url = 'https://github.com/yuvnahr/deepcarv.git'
    os.system(f'git clone --depth 1 --branch {BRANCH_NAME} {clone_url} {REPO_DIR}')
else:
    print('Repo already present:', REPO_DIR)

# Add the package root (where src/ lives) to sys.path
pkg_root = str(REPO_DIR)
if pkg_root not in sys.path:
    sys.path.insert(0, pkg_root)

# Tell paths.py we are in Kaggle runtime
os.environ['KAGGLE_RUNTIME'] = '1'

print('sys.path[0]:', sys.path[0])
print('src exists :', (REPO_DIR / 'src').exists())
print('benchmarks :', (REPO_DIR / 'benchmarks' / 'ByteNet').exists())

gc.collect()


## Cell 3 — Download & Extract FFT-75 Dataset from Google Drive

Downloads `FFT-75.zip` directly to `/kaggle/working/data/` using `gdown`, then extracts it.
**Run this cell once** — it skips the download if the dataset is already present.


In [ ]:
import gc
import gdown
import zipfile
from pathlib import Path

# ── Google Drive file ID (from share link) ────────────────────────────────
GDRIVE_FILE_ID  = '1W3sBVeEPSPAzMgbegHYnMvloxwGigsfS'
# ─────────────────────────────────────────────────────────────────────────

WORKING   = Path('/kaggle/working')
DATA_ROOT = WORKING / 'data'
FFT75_DIR = DATA_ROOT / 'FFT-75'
ZIP_PATH  = DATA_ROOT / 'FFT-75.zip'

DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ── Skip download + extraction if dataset already present ─────────────────
def _dataset_ready():
    for split in ('train', 'val', 'test'):
        if not (FFT75_DIR / '512' / f'{split}.npz').exists():
            return False
    return True

if _dataset_ready():
    print('✅  Dataset already present — skipping download.')
else:
    # Download
    if not ZIP_PATH.exists():
        print(f'Downloading FFT-75.zip from Google Drive (file id: {GDRIVE_FILE_ID}) …')
        url = f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}'
        gdown.download(url, str(ZIP_PATH), quiet=False, fuzzy=True)
        print(f'Downloaded → {ZIP_PATH}  ({ZIP_PATH.stat().st_size / 1e9:.2f} GB)')
    else:
        print(f'ZIP already present at {ZIP_PATH} — skipping download.')

    # Extract
    print('Extracting …')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_ROOT)
    print('Extraction complete.')

    # Remove zip to free disk space
    ZIP_PATH.unlink(missing_ok=True)
    print('ZIP removed to free disk space.')

    # Re-check
    if not _dataset_ready():
        raise FileNotFoundError(
            f'Dataset extraction did not produce the expected layout under {FFT75_DIR}.\n'
            'Expected: FFT-75/512/{{train,val,test}}.npz'
        )
    print('✅  Dataset ready.')

# Free any stray refs
gc.collect()

# ── Quick size report ─────────────────────────────────────────────────────
import os
total_bytes = sum(
    f.stat().st_size
    for f in FFT75_DIR.rglob('*.npz')
    if f.is_file()
)
print(f'\nTotal NPZ size on disk : {total_bytes / 1e9:.2f} GB')
print('Files:')
for f in sorted(FFT75_DIR.rglob('*.npz')):
    print(f'  {f.relative_to(DATA_ROOT)}  ({f.stat().st_size / 1e6:.1f} MB)')


## Cell 4 — Configure Paths & Verify Dataset

Sets up the canonical path variables used by all subsequent cells and runs the
framework's dataset integrity check (shape, dtype, class consistency).


In [ ]:
import gc
from pathlib import Path

WORKING   = Path('/kaggle/working')

# ── Frozen benchmark settings ─────────────────────────────────────────────
FRAGMENT_SIZE = 512                # FROZEN — Scenario #1
VARIANT       = 'bytenet_resnet'  # 'bytenet_resnet' | 'bytenet_former'

# ── Paths (all writeable under /kaggle/working/) ──────────────────────────
FFT75_DIR   = WORKING / 'data' / 'FFT-75'
CKPT_DIR    = WORKING / 'checkpoints'
OUTPUTS_DIR = WORKING / 'outputs'
LOGS_DIR    = WORKING / 'logs'

for d in [CKPT_DIR, OUTPUTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Paths configured:')
for name, p in [('FFT75_DIR', FFT75_DIR), ('CKPT_DIR', CKPT_DIR),
                ('OUTPUTS_DIR', OUTPUTS_DIR), ('LOGS_DIR', LOGS_DIR)]:
    print(f'  {name:<15} {p}')

# ── Dataset structure pre-check ───────────────────────────────────────────
print('\nDataset structure check:')
all_ok = True
for split in ['train', 'val', 'test']:
    npz = FFT75_DIR / str(FRAGMENT_SIZE) / f'{split}.npz'
    status = '✅' if npz.exists() else '❌ MISSING'
    print(f'  FFT-75/{FRAGMENT_SIZE}/{split}.npz  {status}')
    if not npz.exists():
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        f'Dataset not found at {FFT75_DIR}\n'
        'Run Cell 3 first to download and extract the dataset.'
    )

# ── Framework integrity check ─────────────────────────────────────────────
from src.data.verify_dataset import verify_dataset
verify_dataset(data_dir=FFT75_DIR, fragment_size=FRAGMENT_SIZE)
print('\nDataset verification PASSED.')

gc.collect()


## Cell 5 — Sanity Training (2 epochs · 2 000 samples)

Quick end-to-end smoke-test before committing GPU time to the full run.
Uses a small batch size and 2 000-sample subset to verify the entire pipeline.


In [ ]:
import gc
import torch

# Free any leftover GPU memory from previous cells
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

from benchmarks.ByteNet.scripts.train import main as bytenet_train_main

SANITY_CKPT = CKPT_DIR / f'sanity_bytenet_{VARIANT}_{FRAGMENT_SIZE}b.pt'

bytenet_train_main([
    '--data_dir',        str(FFT75_DIR),
    '--fragment_size',   str(FRAGMENT_SIZE),
    '--variant',         VARIANT,
    '--epochs',          '2',
    '--batch_size',      '64',
    '--seed',            '42',
    '--checkpoint_path', str(SANITY_CKPT),
    '--run_dir',         str(OUTPUTS_DIR / 'bytenet_sanity'),
    '--sanity',
])

# Free GPU + CPU memory after sanity run before full training
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print('\nSanity run complete. GPU cache cleared.')


## Cell 6 — Full Training (50 epochs · warmup + cosine · CutMix/Mixup · AMP)

Paper targets:
- **ByteResNet**: 71.0% (512B) | 82.1% (4096B)
- **ByteFormer**: 73.2% (512B) | 81.9% (4096B)

**RAM notes (Kaggle 30 GB limit)**:
- FFT-75/512 train split ≈ 2.5 GB in RAM (stored as int16 tensors).
- ByteResNet model ≈ 50 MB; optimizer state ≈ 100 MB.
- AMP (fp16) halves VRAM during forward/backward passes.
- DataLoader uses `num_workers=0` (data already in RAM cache) — no extra worker processes.


In [ ]:
import gc
import torch

# Clear GPU cache before full training run
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

from benchmarks.ByteNet.scripts.train import main as bytenet_train_main

BEST_CKPT = CKPT_DIR / f'best_bytenet_{VARIANT}_{FRAGMENT_SIZE}b.pt'
RUN_DIR   = OUTPUTS_DIR / f'bytenet_{VARIANT}_{FRAGMENT_SIZE}b'

bytenet_train_main([
    '--data_dir',        str(FFT75_DIR),
    '--fragment_size',   str(FRAGMENT_SIZE),
    '--variant',         VARIANT,
    '--epochs',          '50',
    '--batch_size',      '512',
    '--lr',              '5e-4',
    '--seed',            '42',
    '--checkpoint_path', str(BEST_CKPT),
    '--run_dir',         str(RUN_DIR),
])

# Free GPU memory before evaluation
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print('\nTraining complete. GPU cache cleared.')


## Cell 7 — Evaluation on Frozen Test Set


In [ ]:
import gc
import json
import torch

# Clear GPU memory before loading checkpoint for evaluation
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

from benchmarks.ByteNet.scripts.evaluate import main as bytenet_eval_main

EVAL_OUT = OUTPUTS_DIR / f'bytenet_{VARIANT}_{FRAGMENT_SIZE}b_eval'

bytenet_eval_main([
    '--checkpoint',    str(BEST_CKPT),
    '--data_dir',      str(FFT75_DIR),
    '--fragment_size', str(FRAGMENT_SIZE),
    '--variant',       VARIANT,
    '--out_dir',       str(EVAL_OUT),
    '--batch_size',    '256',
    '--seed',          '42',
])

# Free GPU after evaluation
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# ── Display metrics ────────────────────────────────────────────────────────
with open(EVAL_OUT / 'metrics.json') as f:
    m = json.load(f)
print('\n=== Key Metrics ===')
for k in ['accuracy', 'macro_f1', 'weighted_f1']:
    if k in m:
        print(f'  {k:<35} {m[k]:.4f}')

# Compare to paper target
TARGET = {'bytenet_resnet': 0.710, 'bytenet_former': 0.732}.get(VARIANT, 0.71)
print(f'\n  Paper target (512B, S1) : {TARGET:.3f}')
print(f'  Achieved                : {m.get("accuracy", 0):.4f}')


## Cell 8 — Zip and Save Outputs


In [ ]:
import gc
import shutil

bundle_dir = WORKING / f'ByteNet_{VARIANT}_{FRAGMENT_SIZE}b_bundle'
bundle_dir.mkdir(exist_ok=True)

# Copy evaluation outputs
for fname in ['metrics.json', 'confusion_matrix.csv', 'per_class_metrics.csv',
              'predictions.csv', 'summary.json', 'classification_report.txt']:
    src = EVAL_OUT / fname
    if src.exists():
        shutil.copy2(src, bundle_dir / fname)

# Copy training curves
curve = RUN_DIR / 'training_curves.png'
if curve.exists():
    shutil.copy2(curve, bundle_dir / 'training_curves.png')

# Copy best checkpoint
if BEST_CKPT.exists():
    shutil.copy2(BEST_CKPT, bundle_dir / BEST_CKPT.name)

# Zip
zip_path = WORKING / f'ByteNet_{VARIANT}_{FRAGMENT_SIZE}b.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', bundle_dir)

print(f'Output bundle → {zip_path}')
print(f'Size          : {zip_path.stat().st_size / 1e6:.1f} MB')
print('\nFiles in bundle:')
for f in sorted(bundle_dir.iterdir()):
    print(f'  {f.name}')

gc.collect()
